# Amazon Product Dataset Setup

This notebook downloads the Kaggle dataset `piyushjain16/amazon-product-data` into a local project folder, then loads it for quick inspection and summary statistics.

## Kaggle API Setup (One-Time)

If you see `Kaggle credentials not found at ~/.kaggle/kaggle.json`, do this once:

1. Sign in to Kaggle: https://www.kaggle.com
2. Open Account settings: https://www.kaggle.com/settings/account
3. In **API**, click **Create Legacy API Key** (downloads `kaggle.json`).
4. In a terminal, run:

```bash
mkdir -p ~/.kaggle
mv ~/Downloads/kaggle.json ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json
```

5. Verify file exists (either location works in this notebook):

```bash
ls -l ~/.kaggle/kaggle.json
ls -l ./.kaggle/kaggle.json
```

6. Re-run this notebook.


In [2]:
%pip install -q --upgrade kaggle pandas

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires pandas<3,>=1.3.0, but you have pandas 3.0.1 which is incompatible.


In [3]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import pandas as pd

# CHANGE ME
DATASET_SLUG = "piyushjain16/amazon-product-data"





DATA_DIR = Path("data/raw/" + DATASET_SLUG.split("/")[-1])
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle authentication is required.
home_kaggle = Path.home() / ".kaggle" / "kaggle.json"
project_kaggle = Path.cwd() / ".kaggle" / "kaggle.json"

if home_kaggle.exists():
    os.environ["KAGGLE_CONFIG_DIR"] = str(home_kaggle.parent)
elif project_kaggle.exists():
    os.environ["KAGGLE_CONFIG_DIR"] = str(project_kaggle.parent)
    print(f"Using Kaggle credentials from: {project_kaggle}")
else:
    print("Kaggle credentials not found.")
    print("Expected one of:")
    print(f"- {home_kaggle}")
    print(f"- {project_kaggle}")
    raise FileNotFoundError("Create kaggle.json and rerun this cell.")

kaggle_executable = shutil.which("kaggle")
if kaggle_executable:
    download_cmd = [
        kaggle_executable,
        "datasets",
        "download",
        "-d",
        DATASET_SLUG,
        "-p",
        str(DATA_DIR),
        "--unzip",
    ]
else:
    # Fallback for environments where only the python package is exposed.
    download_cmd = [
        sys.executable,
        "-m",
        "kaggle.cli",
        "datasets",
        "download",
        "-d",
        DATASET_SLUG,
        "-p",
        str(DATA_DIR),
        "--unzip",
    ]

csv_files = sorted(DATA_DIR.rglob("*.csv"))
if not csv_files:
    print(f"Downloading {DATASET_SLUG} to {DATA_DIR} ...")
    result = subprocess.run(download_cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Kaggle download failed:\n{result.stderr or result.stdout}")
    csv_files = sorted(DATA_DIR.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR}.")

dataset_path = csv_files[0]
print(f"Using dataset file: {dataset_path}")

c:\Users\tehwa\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\tehwa\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Using dataset file: data\raw\amazon-product-data\dataset\train.csv


In [4]:
df = pd.read_csv(dataset_path)

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head(10))

Rows: 2,249,698 | Columns: 6


,PRODUCT_ID,TITLE,BULLET_POINTS,DESCRIPTION,PRODUCT_TYPE_ID,PRODUCT_LENGTH
0,1925202,ArtzFolio Tulip Flowers Blackout Curtain for D...,[LUXURIOUS & APPEALING: Beautiful custom-made ...,NaN,1650,2125.980000
1,2673191,Marks & Spencer Girls' Pyjama Sets T86_2561C_N...,"[Harry Potter Hedwig Pyjamas (6-16 Yrs),100% c...",NaN,2755,393.700000
2,2765088,PRIKNIK Horn Red Electric Air Horn Compressor ...,"[Loud Dual Tone Trumpet Horn, Compatible With ...","Specifications: Color: Red, Material: Aluminiu...",7537,748.031495
3,1594019,ALISHAH Women's Cotton Ankle Length Leggings C...,[Made By 95%cotton and 5% Lycra which gives yo...,AISHAH Women's Lycra Cotton Ankel Leggings. Br...,2996,787.401574
4,283658,The United Empire Loyalists: A Chronicle of th...,NaN,NaN,6112,598.424000
5,2152929,HINS Metal Bucket Shape Plant Pot for Indoor &...,"[Simple and elegant, great for displaying indo...",HINS Brings you the most Elegant Looking Pot w...,5725,950.000000
6,413758,Ungifted: My Life and Journey,NaN,NaN,23,598.000000
7,2026580,Delavala Self Adhesive Kitchen Backsplash Wall...,[HIGH QUALITY PVC MATERIAL: The kitchen alumin...,<p><strong>Aluminum Foil Stickers-good kitchen...,6030,984.251967
8,2050239,PUMA Cali Sport Clean Women's Sneakers White L...,[Style Name:-Cali Sport Clean Women's Sneakers...,NaN,3302,393.700000
9,2998633,Hexwell Essential oil for Home Fragrance Oil A...,[100% Pure And Natural Essential Oil Or Fragra...,"Transform your home, workplace or hotel room i...",8201,393.700787


In [5]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null_count": df.notna().sum(),
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(dropna=True),
})

summary["sample_values"] = df.apply(
    lambda s: ", ".join(s.dropna().astype(str).unique()[:3])
)

numeric_stats = df.describe(include=["number"]).T
summary = summary.join(numeric_stats, how="left")

display(summary)

,dtype,non_null_count,null_count,null_pct,unique_count,sample_values,count,mean,std,min,25%,50%,75%,max
PRODUCT_ID,int64,2249698,0,0.00,2249698,"1925202, 2673191, 2765088",2249698.0,1.499795e+06,8.661944e+05,1.0,749479.500000,1499557.5,2.250664e+06,2.999999e+06
TITLE,str,2249685,13,0.00,2210761,ArtzFolio Tulip Flowers Blackout Curtain for D...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BULLET_POINTS,str,1412332,837366,37.22,965329,[LUXURIOUS & APPEALING: Beautiful custom-made ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DESCRIPTION,str,1092316,1157382,51.45,745274,"Specifications: Color: Red, Material: Aluminiu...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PRODUCT_TYPE_ID,int64,2249698,0,0.00,12907,"1650, 2755, 7537",2249698.0,4.000456e+03,3.966146e+03,0.0,230.000000,2916.0,6.403000e+03,1.342000e+04
PRODUCT_LENGTH,float64,2249698,0,0.00,16655,"2125.98, 393.7, 748.0314953",2249698.0,4.071839e+03,1.351685e+06,1.0,511.811023,663.0,1.062992e+03,1.885801e+09
